# Knowledgebase RAG

turn repo to a knowledgebase

In [12]:
import os
import sys

import numpy as np

run_env = os.getenv('RUN_ENV', 'LOCAL')
if run_env == 'COLLAB':
  from google.colab import drive
  ROOT_DIR = '/content/drive'
  drive.mount(ROOT_DIR)
  print('Google drive connected')
  root_data_dir = os.path.join(ROOT_DIR, 'MyDrive', 'ml_course_data')
  lib_path = os.path.join(ROOT_DIR, 'MyDrive', 'src')
  if not os.path.exists(lib_path):
    raise RuntimeError('Upload and `src` dir with code')
  sys.path.append(lib_path)
else:
  repo_dir = os.path.dirname(os.getcwd())  # notebook runs from jupyter_notebooks/
  src_path = os.path.join(repo_dir, 'src')
  if src_path not in sys.path:
    sys.path.append(src_path)
  root_data_dir = os.getenv('DATA_DIR', os.path.join(repo_dir, 'data'))

if not os.path.exists(root_data_dir):
  raise RuntimeError('Data dir not exists')
else:
  print('Data dir content %s: %s' % (root_data_dir, ', '.join(os.listdir(root_data_dir)[:5])))

Data dir content /Users/adzhumurat/PycharmProjects/ai_product_engineer/data: scraping, client_segmentation.csv, messages.db, labeled_data_corpus.csv, nebius_site


In [13]:
from rag.connections import ChromaConnection

client = ChromaConnection().client
client

Connecting to ChromaDB service at http://localhost:8000


## Ollama embeddings

`Chroma` - для хранения. А как получить сами эмеббинги? Можно через Ollama

```shell
curl http://localhost:11434/api/embeddings \
  -d '{
    "model": "granite4:350m",
    "prompt": "Your text to embed goes here"
  }' \
  -H "Content-Type: application/json"
```


Локальная модель: https://ollama.com/library/llama3.2:1b

python - как делать запросы

```python
import requests
import json

# Endpoint and model configuration
url = "http://localhost:11434/api/embeddings"
data = {
    "model": "granite4:350m",
    "prompt": "Your text to embed goes here"
}

response = requests.post(url, json=data)
embedding = response.json().get("embedding")
```

Google AI studio - ещё один вариант (this is not works in Russia)

* [docs](https://ai.google.dev/gemini-api/docs/embeddings)
* [google AI studio](https://aistudio.google.com/)__ 

In [16]:
import requests

try:
    from agent.config import EMBEDDING_MODEL_NAME, OLLAMA_HOST
except ImportError:
    from rag.agent.config import EMBEDDING_MODEL_NAME, OLLAMA_HOST

host = OLLAMA_HOST
query = 'что такое антиградиент'
timeout = 120  # first request may need to reload the model, which can take up to a minute

model = EMBEDDING_MODEL_NAME  # same model as ingestion/retrieve (set via EMBEDDING_MODEL env or agent/config.py)

url = host.rstrip("/") + "/api/embeddings"
response = requests.post(
    url,
    json={"model": model, "prompt": query},
    timeout=timeout,
)
response.raise_for_status()
data = response.json()

query_embedding = data.get("embedding") or data.get("embeddings")
print(f'{model}: {np.array(query_embedding).shape}, {query_embedding[:10]}...')

bge-m3: (1024,), [-1.6550254821777344, -1.6542037725448608, -0.6621116995811462, 0.21713553369045258, 0.2996975779533386, -1.2868103981018066, -0.2145836353302002, -0.6089938282966614, 0.6605094075202942, -0.4355761706829071]...


Запрос к Chroma

In [17]:
from rag.connections import pretty_print_chroma


include = ["documents", "metadatas", "distances"]
print('Available collections:', [c.name for c in client.list_collections()])

collection_name = 'docs2'  # md_docs course content, 768-dim (nomic-embed-text)
collection = client.get_collection(collection_name)
limit = 10

result = collection.query(
    query_embeddings=[query_embedding],
    n_results=limit,
    include=include,
)

pretty_print_chroma(result)


Available collections: ['docs2']
[1] 460d7560-9379-4c36-aa96-aabf39d0a6ae — distance: 759.8238
    source: ml_breadth_questions.md
    Если функция потерь зависит от нескольких параметров то градиент - это вектор Каждая компонента показывает, насколько чувствительна ошибка к изменению конкретного параметра The **gradient** is a mathe…
--------------------------------------------------------------------------------
[2] 55a7073c-2e2c-42a2-ba0a-9a5b4c32f368 — distance: 790.9667
    source: lecture_07_cracking_dnn.md
    Numerical evaluation at Шаг градиентного спуска: где — градиенты Проблемы: - Затухающий градиент — в глубоких сетях градиент умножается много раз и стремится к 0. Решение: ReLU, batch norm, residual c…
--------------------------------------------------------------------------------
[3] 1473097a-f268-43b6-b578-864e1a8e6716 — distance: 850.5457
    source: lecture_07_cracking_dnn.md
    RNN improvenents - Long Short-Term Memory (LSTM) - Gated Recurrent Unit (GRU) - Bidirecti

# Full-text search

using chroma

In [18]:
# 1. Split query into non-empty words
words = query.split()

# 2. Build the $or list dynamically
or_conditions = [{"$contains": word} for word in words]

# 3. Format where_document safely (handling single vs multiple words)
if len(or_conditions) > 1:
    where_doc_filter = {"$or": or_conditions}
elif len(or_conditions) == 1:
    where_doc_filter = or_conditions[0]
else:
    where_doc_filter = None
print(f'Filter: {where_doc_filter}')
# 4. Run collection query or get
results = collection.get(
    where_document=where_doc_filter,
    limit=5
)

pretty_print_chroma(results)

Filter: {'$or': [{'$contains': 'что'}, {'$contains': 'такое'}, {'$contains': 'антиградиент'}]}
[1] c009b310-a8a0-4ceb-b7f2-d0163031cf5d
    source: lecture_09_mlops_intro.md
    # Введение в MLOps ![ivi_main](img/ml_dev_design.png) References * genai-platform * ml system design * MLOps maturity model ## Шаг 1: среда разработки Скачать дамп messages.db и положить в директорию …
--------------------------------------------------------------------------------
[2] c3accd70-0975-4c93-b2df-f963cc640848
    source: lecture_02_vol_02_classification.md
    **Trade-off:** Высокий **recall** может увеличить количество **false positives** (будет низкий **precision**). Для измерения способности отлавливать "негативы" вычисляют **Specificity (TNR)**. Высокая…
--------------------------------------------------------------------------------
[3] a86f7aff-cd35-4b93-8247-e91a7eb0107e
    source: lecture_02_vol_02_classification.md
    Когда использовать - False positives (FP) и false negatives (FN) имеют

### Почему full-text поиск через `where_document` кажется нерелевантным

`collection.get(where_document=...)` — это **фильтр, а не поисковый движок**. Три причины, по которым результаты получаются нерелевантными:

1. **Нет ранжирования.** Chroma возвращает совпавшие чанки в произвольном порядке хранения, `limit=5` просто берёт первые попавшиеся — без сортировки по релевантности.
2. **Стоп-слова доминируют в `$or`.** На 2093 чанках: `что` встречается в 230, `такое` — в 15, `антиградиент` — всего в 6. Условие `$or` выполняется для ~240 чанков, и возвращаются 5 случайных из них — все совпали только по слову «что».
3. **Нет TF-IDF/BM25 и морфологии.** Настоящий полнотекстовый поиск повышает вес редких терминов и игнорирует стоп-слова; `$contains` — обычное вхождение подстроки, где «что» и «антиградиент» равнозначны.

Для сравнения: эмбеддинг-поиск ранжирует по семантической близости и сразу находит объяснение антиградиента. Если нужен честный full-text — используйте BM25 (`rank_bm25`) поверх чанков или ZincSearch; фильтр по `$contains` имеет смысл только как доп. условие к векторному поиску (`collection.query(where_document=...)`).

# Weaviate

вместо chroma - проверяем как работает BM-25

In [19]:
#  Weaviate fulltext seacrh (native BM25, ranked server-side)
from rag.connections import WeaviateConnection, pretty_print_weaviate
from weaviate.classes.query import MetadataQuery

weaviate_conn = WeaviateConnection()          # same 5s connectivity pre-check as Chroma
weaviate_collection = weaviate_conn.get_collection('docs2')  # -> class 'Docs2'

# Weaviate BM25 has no stopword filtering: strip them from the query first
# (defined here so the cell works without running the BM25 cell above)
RU_STOPWORDS = {'что', 'такое', 'как', 'и', 'в', 'на', 'по', 'для', 'это', 'the', 'a', 'is', 'of'}

ft_query = ' '.join(w for w in query.split() if w.lower() not in RU_STOPWORDS)
print(f'BM25 query: {ft_query!r}')

weaviate_ft_results = weaviate_collection.query.bm25(
    query=ft_query,
    query_properties=['text'],
    limit=10,
    return_metadata=MetadataQuery(score=True),
)
pretty_print_weaviate(weaviate_ft_results)
print('Search finished')

Connecting to Weaviate service at http://localhost:8080
BM25 query: 'антиградиент'
[1] ed85e0b1-a5ab-4803-87b7-3303193e6df0 — score: 4.001
    source: vol_02_ml_products_03_knowledgebase_rag.md
    ### Почему BM25 в Weaviate возвращает меньше результатов, чем `limit` `limit` — это **верхняя граница, а не цель**: возвращаются только те чанки, в которых есть **токены запроса**. Токенизатор Weaviat…
--------------------------------------------------------------------------------
[2] 607579ae-c1f8-4e65-a147-014c79cb9e86 — score: 3.763
    source: lecture_02_vol_03_trees_bagging_boosting.md
    Для итерации m = 1, 2, ..., M: **Шаг 1:** Вычислить псевдо-остатки (антиградиенты): **Шаг 2:** Обучить дерево на **Шаг 3:** Для каждого листа найти оптимальный вес : где - множество объектов в листе д…
--------------------------------------------------------------------------------
[3] 85083bef-5b15-4b52-ad46-cc8a33361e46 — score: 3.242
    source: lecture_08_vol_01_rag.md
    ANN (Approximate Neares

/var/folders/bd/bg2rzktd7cl7_q4_11m4dx7r0000gn/T/ipykernel_94639/300151716.py:6: ResourceWarning: unclosed <socket.socket fd=102, family=30, type=1, proto=6, laddr=('::1', 62202, 0, 0), raddr=('::1', 8080, 0, 0)>
  weaviate_collection = weaviate_conn.get_collection('docs2')  # -> class 'Docs2'


### Почему BM25 в Weaviate возвращает меньше результатов, чем `limit`

`limit` — это **верхняя граница, а не цель**: возвращаются только те чанки, в которых есть **токены запроса**. Токенизатор Weaviate по умолчанию (`word`) приводит к нижнему регистру, но **не делает стемминг**, поэтому падежные формы русского языка не совпадают с токеном запроса:

- подстроку `антиградиент` содержат 7 чанков (их находит `$contains` в Chroma),
- но точный токен — только 5: формы `антиградиента`, `антиградиенту`, `антиградиенты` не матчатся с токеном `антиградиент`, и эти чанки отбрасываются.

Отсюда `bm25(query='антиградиент', limit=10)` → всего 5 результатов.

Как поднять полноту (recall) для русского языка:

1. Перечислить все словоформы в запросе: `антиградиент антиградиента антиградиенту антиградиенты`;
2. Пересоздать коллекцию с токенизатором `gse` (со стеммингом);
3. Использовать hybrid-поиск — плотный вектор ловит словоформы и семантику.

*Примечание:* `ResourceWarning: unclosed <socket ...>` — соединение Weaviate не закрывается (`client.close()`), т.к. переиспользуется следующей ячейкой; предупреждение безвредно.

Hybrid search

In [20]:
#  Weaviate hybrid search: BM25 (sparse) + dense vector fused server-side
# alpha = 0 -> pure BM25, alpha = 1 -> pure dense, 0.5 -> balanced hybrid
from weaviate.classes.query import MetadataQuery

alpha = 0.5

weaviate_hybrid_results = weaviate_collection.query.hybrid(
    query=ft_query,          # BM25 part (stopwords already stripped)
    vector=query_embedding,  # dense part: bge-m3 embedding of the full query
    alpha=alpha,
    query_properties=['text'],
    limit=10,
    return_metadata=MetadataQuery(score=True),
)
print(f'Hybrid (alpha={alpha}) top-5:')
pretty_print_weaviate(weaviate_hybrid_results)
print('search finished')

Hybrid (alpha=0.5) top-5:
[1] ed85e0b1-a5ab-4803-87b7-3303193e6df0 — score: 0.748
    source: vol_02_ml_products_03_knowledgebase_rag.md
    ### Почему BM25 в Weaviate возвращает меньше результатов, чем `limit` `limit` — это **верхняя граница, а не цель**: возвращаются только те чанки, в которых есть **токены запроса**. Токенизатор Weaviat…
--------------------------------------------------------------------------------
[2] 85083bef-5b15-4b52-ad46-cc8a33361e46 — score: 0.651
    source: lecture_08_vol_01_rag.md
    ANN (Approximate Nearest Neighbors) — поиск ближайших соседей с допустимым приближением. Он ускоряет поиск на больших объёмах данных, но может вернуть немного иной набор результатов, чем полный точный…
--------------------------------------------------------------------------------
[3] 6fe56413-1b20-49bc-a8d8-b11c063f4e36 — score: 0.559
    source: vol_02_ml_products_03_knowledgebase_rag.md
    'Переменная в формуле - т.н. *шаг градиентного спуска*.\n\nГде - вектор градиента

Compare fulltext with hybrid search

